# XAI Contestation: Dataset Inventory and Feasibility Audit

This notebook loads and audits the XAI-FUNGI dataset used for the proposed
IUI 2027 study on user contestation of AI predictions and explanations.

The objectives are to:

1. verify the available files and dataset structure;
2. load and combine the interview transcripts;
3. inspect transcript schemas and metadata;
4. identify the participant groups and explanation-related records;
5. prepare the data for the contestation feasibility audit.

No original dataset files are modified by this notebook.

In [1]:
# ============================================================
# 1. Libraries, configuration, and project paths
# ============================================================

from __future__ import annotations

import json
import logging
import random
import re
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 250)
pd.set_option("display.width", 160)

warnings.filterwarnings("default", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s",
)

LOGGER = logging.getLogger("xai_contestation")

# ------------------------------------------------------------
# Locate the project root automatically
# ------------------------------------------------------------

def find_project_root(start_path: Path | None = None) -> Path:
    """
    Find the project root by searching upward for a directory
    containing the expected data/ folder.
    """
    start = Path.cwd() if start_path is None else Path(start_path)
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. "
        "Expected a parent directory containing data/."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
TRANSCRIPTS_DIR = DATA_DIR / "transcripts"
VISUALIZATION_PDF_DIR = DATA_DIR / "visualization_modifications"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "01_dataset_feasibility_audit"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"
FIGURE_OUTPUT_DIR = OUTPUT_DIR / "figures"

TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Expected dataset files
# ------------------------------------------------------------

EXPECTED_TOP_LEVEL_CSVS = [
    "CODEBOOK.csv",
    "MAXQDA_SUMMARY.csv",
    "PROBLEMS_en.csv",
    "PROBLEMS_RESPONSES.csv",
    "PROBLEMS.csv",
    "QUESTIONS.csv",
    "SLIDES.csv",
    "SURVEY_en.csv",
    "SURVEY.csv",
    "VISUALIZATION_MODIFICATIONS.csv",
]

EXPECTED_TOP_LEVEL_PDFS = [
    "ORIGINAL_VISUALIZATIONS.pdf",
    "ORIGINAL_VISUALIZATIONS_EN.pdf",
]

# ------------------------------------------------------------
# Initial path validation
# ------------------------------------------------------------

required_directories = [
    DATA_DIR,
    TRANSCRIPTS_DIR,
    VISUALIZATION_PDF_DIR,
]

missing_directories = [
    path for path in required_directories if not path.is_dir()
]

if missing_directories:
    missing_text = "\n".join(str(path) for path in missing_directories)
    raise FileNotFoundError(
        f"Required dataset directories are missing:\n{missing_text}"
    )

LOGGER.info("Project root: %s", PROJECT_ROOT)
LOGGER.info("Data directory: %s", DATA_DIR)
LOGGER.info("Output directory: %s", OUTPUT_DIR)

INFO | Project root: /home/jovyan/fast/x_peer_pdt_detecting_contestation_xai
INFO | Data directory: /home/jovyan/fast/x_peer_pdt_detecting_contestation_xai/data
INFO | Output directory: /home/jovyan/fast/x_peer_pdt_detecting_contestation_xai/outputs/01_dataset_feasibility_audit


In [2]:
# ============================================================
# 2. Reusable data-loading and metadata functions
# ============================================================

def read_csv_robust(path: Path) -> pd.DataFrame:
    """
    Load a CSV file using common encodings found in Polish-language data.

    Delimiter detection is delegated to Python's CSV sniffer through
    sep=None and engine='python'.
    """
    encodings = (
        "utf-8-sig",
        "utf-8",
        "cp1250",
        "windows-1250",
        "latin1",
    )

    errors: list[str] = []

    for encoding in encodings:
        try:
            dataframe = pd.read_csv(
                path,
                encoding=encoding,
                sep=None,
                engine="python",
            )

            dataframe.columns = [
                str(column).strip() for column in dataframe.columns
            ]

            return dataframe

        except (UnicodeDecodeError, pd.errors.ParserError) as error:
            errors.append(f"{encoding}: {error}")

    error_details = "\n".join(errors)

    raise ValueError(
        f"Could not read CSV file: {path}\n"
        f"Attempted encodings:\n{error_details}"
    )


def extract_transcript_metadata(path: Path) -> dict[str, Any]:
    """
    Extract only metadata that can be inferred safely from a transcript
    filename such as DR_IT_05.csv or PK_DE_01.csv.
    """
    stem = path.stem
    parts = stem.split("_")

    group_match = re.search(r"_(DE|IT|SSH)_", f"_{stem}_")
    case_match = re.search(r"(\d+)$", stem)

    return {
        "__source_file": path.name,
        "__transcript_id": stem,
        "__source_prefix": parts[0] if parts else pd.NA,
        "__participant_group": (
            group_match.group(1) if group_match else pd.NA
        ),
        "__case_number": (
            int(case_match.group(1)) if case_match else pd.NA
        ),
    }


def summarize_dataframe(
    name: str,
    dataframe: pd.DataFrame,
    source_type: str,
) -> dict[str, Any]:
    """Create a compact summary record for one loaded dataframe."""
    return {
        "name": name,
        "source_type": source_type,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "duplicate_rows": int(dataframe.duplicated().sum()),
        "fully_empty_rows": int(dataframe.isna().all(axis=1).sum()),
    }


def normalise_empty_strings(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Convert empty or whitespace-only string cells to pandas missing values.
    """
    result = dataframe.copy()

    object_columns = result.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in object_columns:
        result[column] = result[column].replace(
            to_replace=r"^\s*$",
            value=pd.NA,
            regex=True,
        )

    return result

In [3]:
# ============================================================
# 3. Dataset file inventory and validation
# ============================================================

top_level_csv_paths = sorted(DATA_DIR.glob("*.csv"))
top_level_pdf_paths = sorted(DATA_DIR.glob("*.pdf"))
transcript_paths = sorted(TRANSCRIPTS_DIR.glob("*.csv"))
visualization_pdf_paths = sorted(
    VISUALIZATION_PDF_DIR.glob("*.pdf")
)

available_top_level_files = {
    path.name
    for path in top_level_csv_paths + top_level_pdf_paths
}

missing_expected_files = sorted(
    set(EXPECTED_TOP_LEVEL_CSVS + EXPECTED_TOP_LEVEL_PDFS)
    - available_top_level_files
)

inventory_records = []

for path in top_level_csv_paths:
    inventory_records.append(
        {
            "category": "top_level_csv",
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_kb": round(path.stat().st_size / 1024, 2),
        }
    )

for path in top_level_pdf_paths:
    inventory_records.append(
        {
            "category": "top_level_pdf",
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_kb": round(path.stat().st_size / 1024, 2),
        }
    )

for path in transcript_paths:
    inventory_records.append(
        {
            "category": "transcript",
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_kb": round(path.stat().st_size / 1024, 2),
        }
    )

for path in visualization_pdf_paths:
    inventory_records.append(
        {
            "category": "visualization_modification",
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_kb": round(path.stat().st_size / 1024, 2),
        }
    )

file_inventory = (
    pd.DataFrame(inventory_records)
    .sort_values(["category", "filename"])
    .reset_index(drop=True)
)

print(f"Top-level CSV files: {len(top_level_csv_paths)}")
print(f"Top-level PDF files: {len(top_level_pdf_paths)}")
print(f"Transcript files: {len(transcript_paths)}")
print(
    "Visualization-modification PDFs: "
    f"{len(visualization_pdf_paths)}"
)

if missing_expected_files:
    LOGGER.warning(
        "Expected files not found: %s",
        missing_expected_files,
    )
else:
    LOGGER.info("All expected top-level files were found.")

display(
    file_inventory.groupby("category", as_index=False).agg(
        file_count=("filename", "count"),
        total_size_kb=("size_kb", "sum"),
    )
)

INFO | All expected top-level files were found.


Top-level CSV files: 10
Top-level PDF files: 2
Transcript files: 38
Visualization-modification PDFs: 25


,category,file_count,total_size_kb
0,top_level_csv,10,460.87
1,top_level_pdf,2,2531.80
2,transcript,38,1855.53
3,visualization_modification,25,31396.89


In [4]:
# ============================================================
# 4. Load top-level CSV tables
# ============================================================

tables: dict[str, pd.DataFrame] = {}
table_loading_records = []

for path in top_level_csv_paths:
    dataframe = read_csv_robust(path)
    dataframe = normalise_empty_strings(dataframe)

    table_key = path.stem.lower()
    tables[table_key] = dataframe

    table_loading_records.append(
        summarize_dataframe(
            name=path.name,
            dataframe=dataframe,
            source_type="top_level_csv",
        )
    )

table_summary = (
    pd.DataFrame(table_loading_records)
    .sort_values("name")
    .reset_index(drop=True)
)

LOGGER.info(
    "Loaded %d top-level CSV tables.",
    len(tables),
)

display(table_summary)

INFO | Loaded 10 top-level CSV tables.


,name,source_type,rows,columns,duplicate_rows,fully_empty_rows
0,CODEBOOK.csv,top_level_csv,82,2,0,0
1,MAXQDA_SUMMARY.csv,top_level_csv,2886,12,0,0
2,PROBLEMS.csv,top_level_csv,3,23,0,0
3,PROBLEMS_RESPONSES.csv,top_level_csv,78,6,0,0
4,PROBLEMS_en.csv,top_level_csv,3,23,0,0
5,QUESTIONS.csv,top_level_csv,19,4,0,0
6,SLIDES.csv,top_level_csv,16,4,0,0
7,SURVEY.csv,top_level_csv,154,23,0,0
8,SURVEY_en.csv,top_level_csv,154,23,0,0
9,VISUALIZATION_MODIFICATIONS.csv,top_level_csv,381,7,0,0


In [5]:
# ============================================================
# 5. Create explicit references to the main tables
# ============================================================

codebook = tables.get("codebook", pd.DataFrame())
maxqda_summary = tables.get("maxqda_summary", pd.DataFrame())

questions = tables.get("questions", pd.DataFrame())
slides = tables.get("slides", pd.DataFrame())

survey = tables.get("survey", pd.DataFrame())
survey_en = tables.get("survey_en", pd.DataFrame())

problems = tables.get("problems", pd.DataFrame())
problems_en = tables.get("problems_en", pd.DataFrame())
problem_responses = tables.get(
    "problems_responses",
    pd.DataFrame(),
)

visualization_modifications = tables.get(
    "visualization_modifications",
    pd.DataFrame(),
)

named_tables = {
    "codebook": codebook,
    "maxqda_summary": maxqda_summary,
    "questions": questions,
    "slides": slides,
    "survey": survey,
    "survey_en": survey_en,
    "problems": problems,
    "problems_en": problems_en,
    "problem_responses": problem_responses,
    "visualization_modifications": visualization_modifications,
}

for name, dataframe in named_tables.items():
    print(
        f"{name:<30} "
        f"rows={len(dataframe):>5}  "
        f"columns={len(dataframe.columns):>3}"
    )

codebook                       rows=   82  columns=  2
maxqda_summary                 rows= 2886  columns= 12
questions                      rows=   19  columns=  4
slides                         rows=   16  columns=  4
survey                         rows=  154  columns= 23
survey_en                      rows=  154  columns= 23
problems                       rows=    3  columns= 23
problems_en                    rows=    3  columns= 23
problem_responses              rows=   78  columns=  6
visualization_modifications    rows=  381  columns=  7


In [6]:
# ============================================================
# 6. Load and combine transcript files
# ============================================================

transcript_frames: list[pd.DataFrame] = []
transcript_loading_records = []

for path in transcript_paths:
    transcript = read_csv_robust(path)
    transcript = normalise_empty_strings(transcript)

    metadata = extract_transcript_metadata(path)

    # Preserve the original row position inside each source file.
    transcript.insert(
        0,
        "__row_in_file",
        np.arange(1, len(transcript) + 1),
    )

    # Add source metadata using reserved column names.
    for position, (column, value) in enumerate(
        metadata.items(),
        start=0,
    ):
        transcript.insert(position, column, value)

    transcript_frames.append(transcript)

    record = summarize_dataframe(
        name=path.name,
        dataframe=transcript,
        source_type="transcript",
    )
    record.update(metadata)
    transcript_loading_records.append(record)

if not transcript_frames:
    raise ValueError(
        f"No transcript CSV files were found in {TRANSCRIPTS_DIR}"
    )

transcripts = pd.concat(
    transcript_frames,
    ignore_index=True,
    sort=False,
)

transcript_file_summary = (
    pd.DataFrame(transcript_loading_records)
    .sort_values(
        ["__participant_group", "__transcript_id"],
        na_position="last",
    )
    .reset_index(drop=True)
)

LOGGER.info(
    "Loaded %d transcript files with %d combined rows.",
    len(transcript_paths),
    len(transcripts),
)

print(f"Combined transcript shape: {transcripts.shape}")

display(transcript_file_summary.head(10))
display(transcripts.head())

INFO | Loaded 38 transcript files with 14835 combined rows.


Combined transcript shape: (14835, 11)


,name,source_type,rows,columns,duplicate_rows,fully_empty_rows,__source_file,__transcript_id,__source_prefix,__participant_group,__case_number
0,PK_DE_01.csv,transcript,265,11,0,0,PK_DE_01.csv,PK_DE_01,PK,DE,1
1,PK_DE_02.csv,transcript,217,11,0,0,PK_DE_02.csv,PK_DE_02,PK,DE,2
2,PK_DE_03.csv,transcript,113,11,0,0,PK_DE_03.csv,PK_DE_03,PK,DE,3
3,PK_DE_04.csv,transcript,217,11,0,0,PK_DE_04.csv,PK_DE_04,PK,DE,4
4,PK_DE_05.csv,transcript,189,11,0,0,PK_DE_05.csv,PK_DE_05,PK,DE,5
5,PK_DE_06.csv,transcript,110,11,0,0,PK_DE_06.csv,PK_DE_06,PK,DE,6
6,PK_DE_07.csv,transcript,150,11,0,0,PK_DE_07.csv,PK_DE_07,PK,DE,7
7,PK_DE_08.csv,transcript,96,11,0,0,PK_DE_08.csv,PK_DE_08,PK,DE,8
8,PK_DE_09.csv,transcript,146,11,0,0,PK_DE_09.csv,PK_DE_09,PK,DE,9
9,PK_DE_10.csv,transcript,106,11,0,0,PK_DE_10.csv,PK_DE_10,PK,DE,10


,__source_file,__transcript_id,__source_prefix,__participant_group,__case_number,__row_in_file,speaker_id,slide_id,question_id,problem_id,text
0,DR_IT_05.csv,DR_IT_05,DR,IT,5,1,DR,NaN,NaN,NaN,"Transkrypcja została włączona i ona sobie tutaj to spotkanie będzie sobie szło w tle ja z minimalizuję okno, żeby ono pani nie rozpraszało. Jednocześnie włączę tryb nagrywania w telefonie."
1,DR_IT_05.csv,DR_IT_05,DR,IT,5,2,DR,NaN,NaN,NaN,Tak mamy.
2,DR_IT_05.csv,DR_IT_05,DR,IT,5,3,DR,NaN,NaN,NaN,"I zrobimy może w ten sposób, że ja panią zaproszę tutaj na to stanowisko."
3,DR_IT_05.csv,DR_IT_05,DR,IT,5,4,DR,NaN,NaN,NaN,Około pani.
4,DR_IT_05.csv,DR_IT_05,DR,IT,5,5,DR,NaN,NaN,NaN,Wszystko jest w porządku.


In [7]:
# ============================================================
# 7. Initial transcript and schema audit
# ============================================================

transcript_group_summary = (
    transcripts.groupby(
        "__participant_group",
        dropna=False,
    )
    .agg(
        transcript_files=("__source_file", "nunique"),
        transcript_rows=("__source_file", "size"),
    )
    .reset_index()
    .sort_values("__participant_group")
)

transcript_schema = pd.DataFrame(
    {
        "column": transcripts.columns,
        "dtype": [
            str(transcripts[column].dtype)
            for column in transcripts.columns
        ],
        "non_null_count": [
            int(transcripts[column].notna().sum())
            for column in transcripts.columns
        ],
        "missing_count": [
            int(transcripts[column].isna().sum())
            for column in transcripts.columns
        ],
        "unique_values": [
            int(transcripts[column].nunique(dropna=True))
            for column in transcripts.columns
        ],
    }
)

transcript_schema["missing_percent"] = (
    transcript_schema["missing_count"]
    / len(transcripts)
    * 100
).round(2)

print("Transcript files and rows by participant group:")
display(transcript_group_summary)

print("Combined transcript schema:")
display(transcript_schema)

Transcript files and rows by participant group:


,__participant_group,transcript_files,transcript_rows
0,DE,12,1960
1,IT,8,3445
2,SSH,18,9430


Combined transcript schema:


,column,dtype,non_null_count,missing_count,unique_values,missing_percent
0,__source_file,object,14835,0,38,0.00
1,__transcript_id,object,14835,0,38,0.00
2,__source_prefix,object,14835,0,5,0.00
3,__participant_group,object,14835,0,3,0.00
4,__case_number,int64,14835,0,12,0.00
5,__row_in_file,int64,14835,0,1332,0.00
6,speaker_id,object,14835,0,44,0.00
7,slide_id,object,1117,13718,18,92.47
8,question_id,object,558,14277,17,96.24
9,problem_id,object,78,14757,3,99.47


In [8]:
# ============================================================
# 8. Check transcript schema consistency
# ============================================================

schema_by_file = {}

for path in transcript_paths:
    dataframe = read_csv_robust(path)

    schema_by_file[path.name] = tuple(
        str(column).strip()
        for column in dataframe.columns
    )

schema_groups: dict[tuple[str, ...], list[str]] = {}

for filename, schema in schema_by_file.items():
    schema_groups.setdefault(schema, []).append(filename)

schema_group_records = []

for schema_id, (schema, filenames) in enumerate(
    schema_groups.items(),
    start=1,
):
    schema_group_records.append(
        {
            "schema_id": schema_id,
            "file_count": len(filenames),
            "files": ", ".join(filenames),
            "column_count": len(schema),
            "columns": " | ".join(schema),
        }
    )

schema_group_summary = pd.DataFrame(schema_group_records)

print(
    "Number of distinct transcript schemas:",
    len(schema_group_summary),
)

display(schema_group_summary)

Number of distinct transcript schemas: 1


,schema_id,file_count,files,column_count,columns
0,1,38,"DR_IT_05.csv, DR_SSH_01.csv, DR_SSH_02.csv, DR_SSH_03.csv, DR_SSH_04.csv, DR_SSH_06.csv, DR_SSH_07.csv, MK_IT_03.csv, MK_IT_06.csv, MK_SSH_02.csv, MK_SSH_04.csv, MK_SSH_05.csv, MK_SSH_07.csv, MW_IT_02.csv, MW_IT_06.csv, MW_IT_07.csv, MW_SSH_01.cs...",5,speaker_id | slide_id | question_id | problem_id | text


In [9]:
# ============================================================
# 9. Assemble the loaded dataset
# ============================================================

dataset = {
    "project_root": PROJECT_ROOT,
    "data_directory": DATA_DIR,
    "tables": tables,
    "transcripts": transcripts,
    "transcript_file_summary": transcript_file_summary,
    "transcript_schema": transcript_schema,
    "top_level_pdfs": top_level_pdf_paths,
    "visualization_modification_pdfs": visualization_pdf_paths,
    "file_inventory": file_inventory,
}

print("Dataset loaded successfully.")
print(f"Top-level tables: {len(dataset['tables'])}")
print(
    "Transcript files:",
    dataset["transcript_file_summary"]["__source_file"].nunique(),
)
print(f"Combined transcript rows: {len(dataset['transcripts'])}")

Dataset loaded successfully.
Top-level tables: 10
Transcript files: 38
Combined transcript rows: 14835


In [10]:
# ============================================================
# 10. Save dataset inventory and loading summaries
# ============================================================

file_inventory.to_csv(
    TABLE_OUTPUT_DIR / "file_inventory.csv",
    index=False,
)

table_summary.to_csv(
    TABLE_OUTPUT_DIR / "top_level_table_summary.csv",
    index=False,
)

transcript_file_summary.to_csv(
    TABLE_OUTPUT_DIR / "transcript_file_summary.csv",
    index=False,
)

transcript_schema.to_csv(
    TABLE_OUTPUT_DIR / "transcript_schema.csv",
    index=False,
)

schema_group_summary.to_csv(
    TABLE_OUTPUT_DIR / "transcript_schema_groups.csv",
    index=False,
)

LOGGER.info(
    "Audit tables saved to %s",
    TABLE_OUTPUT_DIR,
)

INFO | Audit tables saved to /home/jovyan/fast/x_peer_pdt_detecting_contestation_xai/outputs/01_dataset_feasibility_audit/tables
